In [1]:
!pip install --upgrade transformers datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 20.0 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
dataset = load_dataset("letijo03/sentiment-analysis-taglish-shopee-comment", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


shopee_comment_datasets_annotated.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/50290 [00:00<?, ? examples/s]

In [4]:
label_map = {
    "0": 0,
    "1": 1,
    "2": 2,
    "1 (Contradiction)": 1
}

def clean_label(example):
    example["label"] = label_map[str(example["label"])]
    return example

dataset = dataset.map(clean_label)

Map:   0%|          | 0/50290 [00:00<?, ? examples/s]

In [5]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = dataset["train"]
test_ds  = dataset["test"]

In [6]:
model_name = "dost-asti/RoBERTa-tl-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [7]:
def tokenize(batch):
    texts = []

    for t in batch["Comment"]:
        if t is None:
            texts.append("")
        else:
            texts.append(str(t))

    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [8]:
train_tokenized = train_ds.map(tokenize, batched=True)
test_tokenized  = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/40232 [00:00<?, ? examples/s]

Map:   0%|          | 0/10058 [00:00<?, ? examples/s]

In [9]:
train_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [10]:
training_args = TrainingArguments(
    output_dir="./sentiment_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    logging_steps=200,
    report_to="none")

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    tokenizer=tokenizer
)

trainer.train()

/tmp/ipython-input-4167395953.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
200,0.618000
400,0.425100
600,0.396500
800,0.384400
1000,0.369400
1200,0.350900
1400,0.345000
1600,0.325000
1800,0.351300
2000,0.330500


TrainOutput(global_step=7545, training_loss=0.28978921398569846, metrics={'train_runtime': 3478.6273, 'train_samples_per_second': 34.696, 'train_steps_per_second': 2.169, 'total_flos': 7939184266524672.0, 'train_loss': 0.28978921398569846, 'epoch': 3.0})

In [12]:
test_tokenized

Dataset({
    features: ['Username', 'Rating', 'Date and Time', 'Comment', 'label', 'input_ids', 'attention_mask'],
    num_rows: 10058
})

In [13]:
pred = trainer.predict(test_tokenized)

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_true = pred.label_ids

y_pred = np.argmax(pred.predictions, axis=1)

acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Negative", "Neutral", "Positive"]
))

Accuracy: 0.8819
Confusion Matrix:
[[ 667  126  150]
 [ 173  377  362]
 [ 125  252 7826]]
Classification Report:
              precision    recall  f1-score   support

    Negative       0.69      0.71      0.70       943
     Neutral       0.50      0.41      0.45       912
    Positive       0.94      0.95      0.95      8203

    accuracy                           0.88     10058
   macro avg       0.71      0.69      0.70     10058
weighted avg       0.88      0.88      0.88     10058



In [15]:
trainer.save_model("./sentiment_roberta_taglish")
tokenizer.save_pretrained("./sentiment_roberta_taglish")

('./sentiment_roberta_taglish/tokenizer_config.json',
 './sentiment_roberta_taglish/special_tokens_map.json',
 './sentiment_roberta_taglish/vocab.json',
 './sentiment_roberta_taglish/merges.txt',
 './sentiment_roberta_taglish/added_tokens.json',
 './sentiment_roberta_taglish/tokenizer.json')

In [16]:
!zip -r sentiment_roberta_taglish.zip sentiment_roberta_taglish

  adding: sentiment_roberta_taglish/ (stored 0%)
  adding: sentiment_roberta_taglish/model.safetensors (deflated 7%)
  adding: sentiment_roberta_taglish/tokenizer.json (deflated 82%)
  adding: sentiment_roberta_taglish/config.json (deflated 52%)
  adding: sentiment_roberta_taglish/special_tokens_map.json (deflated 85%)
  adding: sentiment_roberta_taglish/tokenizer_config.json (deflated 75%)
  adding: sentiment_roberta_taglish/training_args.bin (deflated 54%)
  adding: sentiment_roberta_taglish/vocab.json (deflated 59%)
  adding: sentiment_roberta_taglish/merges.txt (deflated 54%)


In [17]:
from google.colab import files
files.download("sentiment_roberta_taglish.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>